# SRF — Terminal Colab

Sistema de Restauração Florestal via terminal web.

**Configuração (uma vez só):**
1. Clique no ícone de chave 🔑 na barra lateral
2. Adicione um segredo chamado `SRF_PASSWORD` com a senha que você recebeu
3. Execute as 3 células abaixo em ordem

- **Célula 1**: Baixa o código e instala dependências
- **Célula 2**: Inicia o servidor e cria link público temporário
- **Célula 3**: Abre o terminal SRF no notebook

In [ ]:
#@title 1. Instalar SRF
import os, subprocess

PROJECT_DIR = '/content/srf'
REPO_URL = 'https://github.com/xAngryBadger/pip-forest.git'

if os.path.exists(PROJECT_DIR):
    print(f'[OK] Repositorio ja existe em {PROJECT_DIR}')
else:
    print(f'[...] Clonando {REPO_URL} ...')
    r = subprocess.run(
        ['git', 'clone', REPO_URL, PROJECT_DIR],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError(f'git clone falhou (codigo {r.returncode})')
    print('[OK] Clone concluido')
subprocess.run(['git', 'checkout', 'refactor/v8'], cwd=PROJECT_DIR, capture_output=True, text=True)

print('[...] Instalando dependencias ...')
r1 = subprocess.run(
    ['pip', 'install', '-q', '-r', os.path.join(PROJECT_DIR, 'requirements-web.txt')],
    capture_output=True, text=True
)
r2 = subprocess.run(
    ['pip', 'install', '-q', 'pandas', 'openpyxl', 'rich', 'colorama'],
    capture_output=True, text=True
)
if r1.returncode != 0 or r2.returncode != 0:
    print(r1.stderr)
    print(r2.stderr)
    raise RuntimeError('pip install falhou')
print('[OK] Dependencias instaladas')

In [ ]:
#@title 2. Iniciar Servidor + Tunnel
import os, time, re, subprocess, requests
from google.colab import userdata

pw = userdata.get('SRF_PASSWORD')
if not pw:
    raise RuntimeError('Adicione SRF_PASSWORD nos segredos do Colab (icone de chave)')

os.environ['SRF_PASSWORD'] = pw
os.environ['SRF_DATA_DIR'] = '/content/srf/data'

PROJECT_DIR = '/content/srf'
TUNNEL_URL_FILE = '/tmp/srf_tunnel_url.txt'
CF_LOG = '/tmp/srf_cloudflared.log'
UVICORN_LOG = '/tmp/srf_uvicorn.log'
PORT = 8000

for name in ['uvicorn', 'cloudflared']:
    subprocess.run(['pkill', '-f', name], capture_output=True)
time.sleep(1)

print('[1/4] Instalando cloudflared ...')
r = subprocess.run(
    ['sudo', 'apt-get', 'install', '-y', 'cloudflared'],
    capture_output=True, text=True
)
if r.returncode != 0:
    print('   apt-get falhou, baixando binario ...')
    cf_bin = '/usr/local/bin/cloudflared'
    r2 = subprocess.run(
        ['curl', '-sL', '-o', cf_bin,
         'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],
        capture_output=True
    )
    if r2.returncode != 0:
        raise RuntimeError('Falha ao baixar cloudflared')
    os.chmod(cf_bin, 0o755)
print('[OK] cloudflared pronto')

print('[2/4] Iniciando uvicorn ...')
uvicorn_log = open(UVICORN_LOG, 'w')
uvicorn_proc = subprocess.Popen(
    ['uvicorn', 'src.web.api:app', '--host', '0.0.0.0', '--port', str(PORT)],
    cwd=PROJECT_DIR,
    stdout=uvicorn_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

server_ready = False
for i in range(30):
    try:
        r = requests.get(f'http://localhost:{PORT}/login', timeout=2)
        if r.status_code == 200:
            server_ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not server_ready:
    uvicorn_log.close()
    print('[ERRO] Uvicorn nao respondeu em 30s')
    print(open(UVICORN_LOG).read()[-2000:])
    raise RuntimeError('Servidor nao iniciou')
print(f'[OK] Uvicorn rodando na porta {PORT}')

print('[3/4] Iniciando Cloudflare Tunnel ...')
if os.path.exists(CF_LOG):
    os.remove(CF_LOG)

cf_log = open(CF_LOG, 'w')
cf_proc = subprocess.Popen(
 ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}', '--http2-origin'],
    stdout=cf_log,
    stderr=subprocess.STDOUT
)

print('[4/4] Aguardando URL do tunnel (ate 60s) ...')
tunnel_url = None
for i in range(20):
    time.sleep(3)
    try:
        with open(CF_LOG) as f:
            log = f.read()
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log)
        if m:
            tunnel_url = m.group(0)
            break
    except Exception:
        pass

if not tunnel_url:
    cf_log.close()
    print('[ERRO] URL do tunnel nao encontrada')
    if os.path.exists(CF_LOG):
        print(open(CF_LOG).read()[-2000:])
    else:
        print('(sem log)')
    raise RuntimeError('Tunnel nao iniciou')

with open(TUNNEL_URL_FILE, 'w') as f:
    f.write(tunnel_url)

print()
print('=' * 50)
print(f'  URL PUBLICA: {tunnel_url}')
print('=' * 50)
print()
print('Execute a celula 3 para abrir o terminal.')

In [ ]:
#@title 3. Terminal SRF
import os
import IPython.display as disp

TUNNEL_URL_FILE = '/tmp/srf_tunnel_url.txt'

if not os.path.exists(TUNNEL_URL_FILE):
    raise RuntimeError('Execute a celula 2 primeiro para iniciar o servidor')

tunnel_url = open(TUNNEL_URL_FILE).read().strip()

print(f'URL: {tunnel_url}')
print('Digite a senha no iframe abaixo para acessar o terminal.')
print(f'Ou abra em nova aba: {tunnel_url}/app')

disp.HTML(f"""
<div style="border:3px solid #2D6A4F;box-shadow:4px 4px 0px #000;border-radius:4px;overflow:hidden;">
<div style="background:#2D6A4F;color:#fff;padding:6px 12px;font-family:monospace;font-size:13px;font-weight:bold;">
SRF Terminal &mdash; <a href="{tunnel_url}/app" style="color:#90EE90;text-decoration:none;" target="_blank">abrir em nova aba</a>
</div>
<iframe src="{tunnel_url}/app" style="width:100%;height:600px;border:0;"></iframe>
</div>
""")
